# Sales Forecast API

## Exploratory Data Analysis (EDA)

### Objective

The purpose of this notebook is to perform an initial exploration of the Rossmann Store Sales dataset.

The goals are:

- Understand the available data
- Identify potential data quality issues
- Perform basic cleaning
- Explore the target variable
- Identify patterns that may be useful for feature engineering
- Detect potential sources of data leakage

This notebook intentionally focuses on the analyses that will support the machine learning pipeline rather than an extensive business analysis.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import RAW_DATA_DIR

plt.style.use("ggplot")
pd.set_option("display.max_columns", None)

## Load Datasets

The competition provides three datasets:

- `train.csv`: historical sales data
- `test.csv`: observations for future predictions
- `store.csv`: metadata describing each store

In [ ]:
train = pd.read_csv(RAW_DATA_DIR / "train.csv")
test = pd.read_csv(RAW_DATA_DIR / "test.csv")
store = pd.read_csv(RAW_DATA_DIR / "store.csv")

In [ ]:
train.head()

In [ ]:
store.head()

## Dataset Overview

Let's inspect the structure of each dataset before performing any transformations.

In [ ]:
print(f"Train shape: {train.shape}")
print(f"Test shape : {test.shape}")
print(f"Store shape: {store.shape}")

In [ ]:
train.info()

In [ ]:
store.info()

## Missing Values

Missing values may require special treatment during feature engineering.

In [ ]:
train.isna().sum().sort_values(ascending=False)

In [ ]:
store.isna().sum().sort_values(ascending=False)

In [ ]:
(
    store.isna()
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
)

Missing values are mostly explained by business rules rather than data quality issues. These values will be handled during feature engineering.

## Duplicate Rows

In [ ]:
print("Train duplicates:", train.duplicated().sum())
print("Store duplicates:", store.duplicated().sum())

## Convert Date Columns

In [ ]:
train["Date"] = pd.to_datetime(train["Date"])
test["Date"] = pd.to_datetime(test["Date"])

## Merge Store Information

The machine learning model will use both transactional and store metadata.

In [ ]:
df = train.merge(store, on="Store", how="left")

In [ ]:
df.head()

## Target Variable

Our prediction target is **Sales**.

In [ ]:
df["Sales"].describe()

In [ ]:
plt.figure(figsize=(8,4))
plt.hist(df["Sales"], bins=50)
plt.title("Sales Distribution")
plt.xlabel("Sales")
plt.ylabel("Frequency")
plt.show()

## Store Closures

Stores that are closed naturally have zero sales.

Understanding this relationship is important because these observations may need special treatment during model training.

In [ ]:
df["Open"].value_counts()

In [ ]:
df.groupby("Open")["Sales"].describe()

## Time Coverage

In [ ]:
print(df["Date"].min())
print(df["Date"].max())

## Total Sales Over Time

In [ ]:
daily_sales = (
    df.groupby("Date")["Sales"]
      .sum()
)

plt.figure(figsize=(15,5))

daily_sales.plot()

plt.title("Daily Sales")
plt.xlabel("Date")
plt.ylabel("Sales")

plt.show()

## Promotions

In [ ]:
df.groupby("Promo")["Sales"].agg(["mean", "median", "count"])

In [ ]:
promo_sales = (
    df.groupby("Promo")["Sales"]
      .mean()
)

plt.figure(figsize=(5,4))

promo_sales.plot(kind="bar")

plt.title("Average Sales by Promotion")

plt.show()

## Store types

In [ ]:
df.groupby("StoreType")["Sales"].agg(["mean", "median", "count"])

## Correlation with Sales

In [ ]:
numeric = df.select_dtypes(include=np.number)

numeric.corr(numeric_only=True)["Sales"].sort_values(ascending=False)

## Customers

The `Customers` feature is strongly correlated with Sales.

However, this information is **not available when forecasting future sales**, meaning it would introduce data leakage if used for training.

The feature will therefore be excluded during model development.

In [ ]:
plt.figure(figsize=(6,6))

plt.scatter(
    df["Customers"],
    df["Sales"],
    alpha=0.2,
    s=3
)

plt.xlabel("Customers")
plt.ylabel("Sales")

plt.show()

# Conclusions

Main observations from the initial exploration:

- The dataset contains historical daily sales for multiple Rossmann stores.
- Store metadata must be merged before feature engineering.
- Missing values exist mainly in competition and promotion-related variables.
- Closed stores consistently present zero sales.
- Sales exhibit strong temporal patterns.
- Promotions are associated with higher average sales.
- The `Customers` feature is highly predictive but cannot be used in production because it is unknown at prediction time.
- Date-derived features will likely play an important role during feature engineering.